# Etapa 4: Score de Risco

Transforma a probabilidade matematica do modelo campeao (Logistic Regression)
em uma metrica de negocios acionavel: o **Score de Risco** (0-100) com **Tiers de Prioridade**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import sys
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
FIGURES_DIR = REPO_ROOT / 'reports' / 'figures'

# Adicionar src/ ao path para importar o modulo de inferencia
sys.path.insert(0, str(REPO_ROOT / 'src'))

## 1. Carregamento dos Dados e Modelo Campeao

In [2]:
# Carregar a base processada e o modelo
df_processed = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'telco_churn_features.csv')
modelo = joblib.load(REPO_ROOT / 'models' / 'campeao.joblib')

print(f'Base processada: {df_processed.shape}')
print(f'Modelo: {type(modelo)}')

Base processada: (7043, 24)
Modelo: <class 'sklearn.pipeline.Pipeline'>


## 2. Calculo do Score de Risco

O Score de Risco e uma transformacao direta e monotonica da probabilidade
emitida pelo `.predict_proba()`: `Risk_Score = round(prob_churn * 100)`.

Nao aplicamos calibracao adicional porque a Logistic Regression ja emite
probabilidades naturalmente bem calibradas (verificado na curva de calibracao da Etapa 3).

In [3]:
# Separar features (dropar Churn que eh o target real)
X = df_processed.drop(columns=['Churn'])
y_real = df_processed['Churn']

# Aplicar predict_proba no dataset inteiro
probabilidades = modelo.predict_proba(X)[:, 1]

# Criar Risk_Score: probabilidade * 100, arredondado para inteiro
df_processed['Risk_Score'] = np.round(probabilidades * 100).astype(int)

print('Estatisticas do Risk_Score:')
print(df_processed['Risk_Score'].describe())
print(f'\nRange: {df_processed["Risk_Score"].min()} a {df_processed["Risk_Score"].max()}')

Estatisticas do Risk_Score:
count    7043.000000
mean       41.827772
std        28.707043
min         2.000000
25%        14.000000
50%        39.000000
75%        69.000000
max        94.000000
Name: Risk_Score, dtype: float64

Range: 2 a 94


## 3. Clusterizacao em Tiers de Risco

| Tier | Faixa | Acao Operacional |
| --- | --- | --- |
| Baixo Risco | 0 a 30 | Monitoramento passivo |
| Risco Medio | 31 a 70 | Campanhas de retencao preventivas |
| Alto Risco | 71 a 100 | Acao imediata (ligacao, oferta, desconto) |

In [4]:
def classificar_tier(score):
    """Atribui faixa de risco com base no score 0-100."""
    if score <= 30:
        return 'Baixo Risco'
    elif score <= 70:
        return 'Risco Medio'
    else:
        return 'Alto Risco'

df_processed['Risk_Tier'] = df_processed['Risk_Score'].apply(classificar_tier)

# Distribuicao dos Tiers
tier_counts = df_processed['Risk_Tier'].value_counts()
tier_pct = df_processed['Risk_Tier'].value_counts(normalize=True) * 100

distribuicao = pd.DataFrame({
    'Clientes': tier_counts,
    '% do Total': tier_pct.round(1)
})
# Ordenar: Baixo, Medio, Alto
distribuicao = distribuicao.loc[['Baixo Risco', 'Risco Medio', 'Alto Risco']]
print('Distribuicao dos Tiers de Risco:')
print(distribuicao.to_string())

Distribuicao dos Tiers de Risco:
             Clientes  % do Total
Risk_Tier                        
Baixo Risco      2927        41.6
Risco Medio      2476        35.2
Alto Risco       1640        23.3


## 4. Validacao de Coerencia do Score (SLA)

In [5]:
# SLA: Clientes com Churn=1 devem ter score medio maior que Churn=0
media_churn_sim = df_processed[df_processed['Churn'] == 1]['Risk_Score'].mean()
media_churn_nao = df_processed[df_processed['Churn'] == 0]['Risk_Score'].mean()

print(f'Score medio (Churn=Sim): {media_churn_sim:.1f}')
print(f'Score medio (Churn=Nao): {media_churn_nao:.1f}')
print(f'\nCoerencia monotonica: {"PASS" if media_churn_sim > media_churn_nao else "FAIL"}')
print(f'Separacao: {media_churn_sim - media_churn_nao:.1f} pontos')

Score medio (Churn=Sim): 67.4
Score medio (Churn=Nao): 32.6

Coerencia monotonica: PASS
Separacao: 34.8 pontos


## 5. Visualizacao da Distribuicao de Risco

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma do Risk_Score por Churn real
axes[0].hist(df_processed[df_processed['Churn']==0]['Risk_Score'],
             bins=30, alpha=0.6, label='Churn=Nao', color='steelblue')
axes[0].hist(df_processed[df_processed['Churn']==1]['Risk_Score'],
             bins=30, alpha=0.6, label='Churn=Sim', color='tomato')
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_ylabel('Frequencia')
axes[0].set_title('Distribuicao do Score de Risco por Churn Real')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Barplot dos Tiers
tier_order = ['Baixo Risco', 'Risco Medio', 'Alto Risco']
cores = ['#2ecc71', '#f39c12', '#e74c3c']
contagem = [tier_counts.get(t, 0) for t in tier_order]
axes[1].bar(tier_order, contagem, color=cores)
axes[1].set_ylabel('Numero de Clientes')
axes[1].set_title('Clientes por Tier de Risco')
axes[1].grid(axis='y', alpha=0.3)

# Adicionar labels nos bars
for i, (t, c) in enumerate(zip(tier_order, contagem)):
    axes[1].text(i, c + 50, f'{c}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'score_distribuicao_risco.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em reports/figures/score_distribuicao_risco.png')

Salvo em reports/figures/score_distribuicao_risco.png


## 6. Demonstracao da Funcao `predict_and_score`

Teste end-to-end: simula o fluxo que o Streamlit executara.
Carrega a base BRUTA original e passa pela funcao portatil.

In [7]:
from inference import predict_and_score

# Carregar a base BRUTA (formato original do Kaggle)
df_bruta = pd.read_csv(REPO_ROOT / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Base bruta carregada: {df_bruta.shape}')

# Rodar o pipeline completo
df_resultado = predict_and_score(df_bruta, modelo)

print(f'\nColuna adicionadas: Churn_Predicted, Risk_Score, Risk_Tier')
print(f'Shape final: {df_resultado.shape}')
print(f'\nAmostra dos resultados (primeiros 10):')
print(df_resultado[['Churn', 'Churn_Predicted', 'Risk_Score', 'Risk_Tier']].head(10).to_string())

Base bruta carregada: (7043, 21)

Coluna adicionadas: Churn_Predicted, Risk_Score, Risk_Tier
Shape final: (7043, 27)

Amostra dos resultados (primeiros 10):
   Churn  Churn_Predicted  Risk_Score    Risk_Tier
0      0                1          74   Alto Risco
1      0                0          16  Baixo Risco
2      1                1          51  Risco Medio
3      0                0          12  Baixo Risco
4      1                1          85   Alto Risco
5      1                1          90   Alto Risco
6      0                1          70  Risco Medio
7      0                0          48  Risco Medio
8      1                1          80   Alto Risco
9      0                0           6  Baixo Risco


## 7. Exportar Base Scored (Auditoria)

In [8]:
# Salvar base completa com scores para auditoria
output_path = REPO_ROOT / 'data' / 'processed' / 'telco_churn_scored.csv'
df_processed.to_csv(output_path, index=False)
print(f'Base scored salva em: {output_path}')
print(f'Shape: {df_processed.shape}')

Base scored salva em: C:\Users\berna\.antigravity\projects\ds-synapsee-challenge\ds-synapsee-challenge\data\processed\telco_churn_scored.csv
Shape: (7043, 26)
